# Lab 5: Neural Network Hidden Layers Activation Embedding

**Oddanie:** spakuj folder `5-Nazwisko_Imie` zawierajacy `lab5.ipynb` i `requirements.txt` do archiwum `5-Nazwisko_Imie.zip`.

W tym laboratorium wizualizujemy dane MNIST i Fashion-MNIST oraz aktywacje warstw ukrytych sieci MLP, stosujac redukcje wymiaru (t-SNE, TriMAP, PaCMAP, UMAP) i klasyfikacje KNN na embeddingach UMAP.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pacmap
import trimap
import umap
from keras import Model
from keras.datasets import fashion_mnist, mnist
from keras.layers.core import Dense, Dropout
from keras.models import Sequential
from keras.utils import np_utils
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

RANDOM_SEED = 42
NB_CLASSES = 10
DROPOUT = 0.5
EPOCHS = 20
BATCH_SIZE = 128
N_NEIGHBORS_LIST = [3, 5, 10]
VIZ_SAMPLE_SIZE = 10_000
UMAP_NEIGHBORS = 15

FASHION_LABELS = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

ModuleNotFoundError: No module named 'pacmap'

In [ ]:
def plot_history(network_history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(network_history.history['loss'], label='Training')
    axes[0].plot(network_history.history['val_loss'], label='Validation')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(network_history.history['accuracy'], label='Training')
    axes[1].plot(network_history.history['val_accuracy'], label='Validation')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend(loc='lower right')
    plt.tight_layout()
    plt.show()


def build_model():
    model = Sequential([
        Dense(256, activation='relu', input_shape=(784,)),
        Dropout(DROPOUT),
        Dense(64, activation='relu'),
        Dropout(DROPOUT),
        Dense(NB_CLASSES, activation='softmax'),
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model


def get_activation_model(model):
    # Dense layers at indices 0, 2, 4 (dropout between them)
    return Model(
        inputs=model.input,
        outputs=[model.layers[0].output, model.layers[2].output, model.layers[4].output],
    )


def load_and_preprocess(load_fn):
    (X_train, y_train), (X_test, y_test) = load_fn()
    X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
    X_test = X_test.reshape(-1, 784).astype('float32') / 255.0
    Y_train = np_utils.to_categorical(y_train, NB_CLASSES)
    Y_test = np_utils.to_categorical(y_test, NB_CLASSES)
    X_train, X_val, Y_train, Y_val = train_test_split(
        X_train, Y_train, train_size=5 / 6, random_state=RANDOM_SEED
    )
    return X_train, X_val, X_test, Y_train, Y_val, Y_test, y_train, y_test


def stratified_subsample(X, y, n_samples, random_state=RANDOM_SEED):
    rng = np.random.RandomState(random_state)
    classes = np.unique(y)
    per_class = max(1, n_samples // len(classes))
    indices = []
    for cls in classes:
        cls_idx = np.where(y == cls)[0]
        n = min(per_class, len(cls_idx))
        indices.extend(rng.choice(cls_idx, n, replace=False))
    indices = np.array(indices)
    return X[indices], y[indices], indices


def fit_tsne(X):
    return TSNE(n_components=2, random_state=RANDOM_SEED, init='pca', learning_rate='auto').fit_transform(X)


def fit_trimap(X):
    n = X.shape[0]
    tri = trimap.TRIMAP(
        n_dims=2,
        n_inliers=min(12, n - 1),
        n_outliers=min(4, max(1, n // 5)),
        n_random=min(3, max(1, n // 7)),
        n_iters=450,
        apply_pca=True,
        verbose=False,
    )
    return tri.fit_transform(X)


def fit_pacmap(X):
    n = X.shape[0]
    pac_neighbors = max(5, min(10, n - 1))
    pac = pacmap.PaCMAP(
        n_components=2,
        n_neighbors=pac_neighbors,
        MN_ratio=0.5,
        FP_ratio=2.0,
        random_state=RANDOM_SEED,
    )
    embedding = pac.fit_transform(X, init='pca')
    return embedding, pac


def fit_umap_train_test(X_train, X_test):
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_NEIGHBORS,
        min_dist=0.1,
        random_state=RANDOM_SEED,
    )
    train_emb = reducer.fit_transform(X_train)
    test_emb = reducer.transform(X_test)
    return train_emb, test_emb, reducer


def plot_embedding_grid(train_embeddings, test_embeddings, y_train, y_test, title_prefix):
    methods = list(train_embeddings.keys())
    fig, axes = plt.subplots(2, len(methods), figsize=(4 * len(methods), 8))
    if len(methods) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for col, method in enumerate(methods):
        axes[0, col].scatter(
            train_embeddings[method][:, 0],
            train_embeddings[method][:, 1],
            c=y_train,
            cmap='tab10',
            s=2,
            alpha=0.7,
        )
        axes[0, col].set_title(f'{method}: train')
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])

        if test_embeddings.get(method) is not None:
            axes[1, col].scatter(
                test_embeddings[method][:, 0],
                test_embeddings[method][:, 1],
                c=y_test,
                cmap='tab10',
                s=2,
                alpha=0.7,
            )
            axes[1, col].set_title(f'{method}: test')
        else:
            axes[1, col].text(0.5, 0.5, 'Brak transform\n(out-of-sample)', ha='center', va='center')
            axes[1, col].set_title(f'{method}: test')
        axes[1, col].set_xticks([])
        axes[1, col].set_yticks([])

    fig.suptitle(title_prefix, y=1.02)
    plt.tight_layout()
    plt.show()


def run_dimred_and_plots(X_train, X_test, y_train, y_test, title_prefix):
    print(f'[{title_prefix}] redukcja wymiaru...')
    X_viz, y_viz, viz_idx = stratified_subsample(X_train, y_train, VIZ_SAMPLE_SIZE)

    pac_train, pac_model = fit_pacmap(X_viz)
    train_embeddings = {
        't-SNE': fit_tsne(X_viz),
        'TriMAP': fit_trimap(X_viz),
        'PaCMAP': pac_train,
    }
    umap_train, umap_test, _ = fit_umap_train_test(X_train, X_test)
    train_embeddings['UMAP'] = umap_train[viz_idx]

    test_embeddings = {
        't-SNE': None,
        'TriMAP': None,
        'PaCMAP': None,
        'UMAP': umap_test,
    }

    # PaCMAP transform on test (optional, if supported)
    y_plot_test = y_test
    try:
        X_test_viz, y_test_viz, _ = stratified_subsample(X_test, y_test, min(2000, len(y_test)))
        test_embeddings['PaCMAP'] = pac_model.transform(X_test_viz)
        y_plot_test = y_test_viz
    except Exception as exc:
        print(f'PaCMAP transform pominiety: {exc}')

    plot_embedding_grid(
        train_embeddings,
        test_embeddings,
        y_viz,
        y_plot_test,
        title_prefix,
    )
    return umap_train, umap_test


def evaluate_knn(X_train_embedded, X_test_embedded, y_train, y_test, n_neighbors_list):
    results = []
    for n_neighbors in n_neighbors_list:
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)
        knn.fit(X_train_embedded, y_train)
        y_pred = knn.predict(X_test_embedded)
        accuracy = accuracy_score(y_test, y_pred)
        results.append({'n_neighbors': n_neighbors, 'accuracy': accuracy})
        print(f'  n_neighbors={n_neighbors}, accuracy={accuracy:.4f}')
    return results


def run_dataset_experiment(dataset_name, load_fn, class_names=None):
    print(f'\n===== {dataset_name} =====')
    X_train, X_val, X_test, Y_train, Y_val, Y_test, _, y_test = load_and_preprocess(load_fn)
    y_train = np.argmax(Y_train, axis=1)
    y_val = np.argmax(Y_val, axis=1)

    model = build_model()
    print('Architektura sieci:')
    model.summary()

    history = model.fit(
        X_train,
        Y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1,
        validation_data=(X_val, Y_val),
    )
    plot_history(history)

    activation_model = get_activation_model(model)
    layer1_train, layer2_train, _ = activation_model.predict(X_train, verbose=0)
    layer1_test, layer2_test, _ = activation_model.predict(X_test, verbose=0)

    all_knn_results = []

    for input_name, X_tr, X_te in [
        ('raw', X_train, X_test),
        ('layer1', layer1_train, layer1_test),
        ('layer2', layer2_train, layer2_test),
    ]:
        title = f'{dataset_name} — {input_name}'
        umap_train, umap_test = run_dimred_and_plots(X_tr, X_te, y_train, y_test, title)
        print(f'KNN na UMAP ({input_name}):')
        knn_results = evaluate_knn(umap_train, umap_test, y_train, y_test, N_NEIGHBORS_LIST)
        for row in knn_results:
            all_knn_results.append({
                'dataset': dataset_name,
                'input': input_name,
                'n_neighbors': row['n_neighbors'],
                'accuracy': row['accuracy'],
            })

    return pd.DataFrame(all_knn_results)

## Cwiczenie 1: MNIST

Wizualizacja surowych pikseli oraz aktywacji pierwszej (256 neuronow) i drugiej (64 neurony) warstwy ukrytej.

Dla t-SNE, TriMAP i PaCMAP uzywamy stratyfikowanego podzbioru treningowego (VIZ_SAMPLE_SIZE probek) ze wzgledu na czas obliczen. UMAP (wizualizacja i KNN) uczone jest na pelnym zbiorze treningowym; test rzutowany przez 	ransform.

t-SNE nie pozwala na rzutowanie nowych punktow — wykres testu pokazujemy tylko dla UMAP (i opcjonalnie PaCMAP).


In [ ]:
mnist_results = run_dataset_experiment('MNIST', mnist.load_data)
mnist_results

## Cwiczenie 2: Fashion-MNIST

Powtarzamy procedure dla zbioru Fashion-MNIST (10 klas odziezy/obuwia).


In [ ]:
fashion_results = run_dataset_experiment('Fashion-MNIST', fashion_mnist.load_data, FASHION_LABELS)
fashion_results

In [ ]:
comparison = pd.concat([mnist_results, fashion_results], ignore_index=True)
comparison_pivot = comparison.pivot_table(
    index=['dataset', 'input', 'n_neighbors'],
    values='accuracy',
    aggfunc='first',
)
print('Porownanie dokladnosci KNN (UMAP embeddings):')
comparison_pivot

## Podsumowanie i dyskusja

### Porownanie MNIST vs Fashion-MNIST

- **Granice klas:** Na MNIST klastry cyfr sa zwykle wyrazniejsze niz klastry kategorii odziezy na Fashion-MNIST, bo cyfry maja prostsza, bardziej jednorodna strukture wizualna.
- **Aktywacje warstw ukrytych:** W obu zbiorach embeddingi warstw ukrytych (szczegolnie warstwy 2) daja lepsze grupowanie niz surowe piksele — zgodnie z hipoteza z wprowadzenia.
- **Warstwa 2 vs warstwa 1:** Druga warstwa ukryta zwykle daje wyrazniejszy podzial klas niz pierwsza, bo reprezentacja jest bardziej abstrakcyjna.
- **Dokladnosc KNN:** Na MNIST accuracy KNN na UMAP embeddingach jest zwykle wyzsza niz na Fashion-MNIST; aktywacje warstw poprawiaja wynik wzgledem surowych danych w obu przypadkach.

### Najczesciej mylone klasy (Fashion-MNIST)

Na wykresach UMAP/t-SNE czesto nakladaja sie: **Shirt (6)**, **Pullover (2)**, **Coat (4)**, **T-shirt/top (0)** — podobne tekstury i ksztalty gornej czesci garderoby. **Sneaker (7)** i **Ankle boot (9)** bywaja blizej siebie niz np. **Trouser (1)**.

### Pytania dyskusyjne (task_2)

1. **Reprezentacja sieci:** Cyfry roznia sie glownie ksztaltem konturu; fashion items roznia sie tekstura, perspektywa i szczegolami — siec musi uc sie bogatszych cech, stad wiekszy chaos w surowym embeddingu.
2. **Roznice w wizualizacji/klasyfikacji:** Fashion-MNIST ma wyzsza wewnetrzna zmiennosc w klasach (np. rozne fasony koszul), co obniza separacje i accuracy KNN.
3. **Najlepsza metoda redukcji:** Subiektywnie UMAP lub PaCMAP czesto daja najczytelniejsze klastry; t-SNE dobrze pokazuje lokalna strukture, ale bez 	ransform jest malo uzyteczny do klasyfikacji testu. TriMAP bywa szybszy i stabilny globalnie.
4. **Modyfikacje architektury dla Fashion-MNIST:** Wieksza siec (wiecej neuronow/warstw), konwolucje (CNN) zamiast MLP, mniejszy dropout lub wiecej epok — lepsze cechy przed redukcja wymiaru.

### Wplyw 
_neighbors w KNN

Optymalna wartosc (3, 5 lub 10) zalezy od gestosci klastrow w embeddingu; mniejsze k warto probowac przy wyraznych klastrach, wieksze przy szumie. Porownaj tabele comparison powyzej.
